# Example: GloVe Embeddings from Scratch
In this example, we train a GloVe (Global Vectors for Word Representation) model on the same toy corpus used in L9a. GloVe learns word embeddings by factorizing a weighted log co-occurrence matrix, combining the strengths of count-based and prediction-based methods. After training, we compute cosine similarities for selected word pairs so the student can compare these results to the CBOW and Skip-Gram embeddings from L9a and L9c.

> __Learning Objectives:__
> 
> By the end of this example, you should be able to:
>
> * __Co-occurrence matrix construction__: Build a weighted co-occurrence matrix from a corpus using distance-based weighting within a sliding window. Understand how closer words receive higher co-occurrence counts through the $1/d$ weighting scheme.
> * __GloVe training__: Train GloVe embeddings by minimizing a weighted least-squares objective over non-zero co-occurrence entries. Monitor the loss curve to verify convergence of the embedding parameters.
> * __Embedding comparison__: Extract final GloVe embeddings by combining word and context vectors into a single representation. Compute cosine similarities for selected word pairs and compare to prediction-based methods such as CBOW and Skip-Gram.

Let's get started!
___

## Setup, Data, and Prerequisites
We set up the computational environment by including the `Include.jl` file, loading any needed resources, such as sample datasets, and setting up any required constants.

> The `Include.jl` file also loads external packages, various functions that we will use in the exercise, and custom types to model the components of our problem. It checks for a `Manifest.toml` file; if it finds one, packages are loaded. Other packages are downloaded and then loaded.

In [ ]:
include(joinpath(@__DIR__, "Include.jl")); # include the Include.jl file

### Data
We use the same toy corpus of simple sentences from the L9a examples. Let's construct the `sentences::Array{String,1}` variable containing our corpus.

In [ ]:
sentences = let

    # initialize -
    sentences_array = Array{String,1}(); # initialize an array of sentence strings

    # add sentences -
    push!(sentences_array, "I love machine learning and data science .");
    push!(sentences_array, "Machine learning is fun .");
    push!(sentences_array, "Machine learning is great .");
    push!(sentences_array, "I love coding machine learning in Julia !");
    push!(sentences_array, "Julia is a great programming language ?");
    push!(sentences_array, "I enjoy learning new things about data science , machine learning , and artificial intelligence .");
    
    # return the array
    sentences_array
end

Next, we build the vocabulary in the `vocabulary::Dict{String, Int64}` variable, where keys are unique words and values are their indices, and the inverse vocabulary in the `inverse_vocabulary::Dict{Int64, String}` variable, where keys are indices and values are words.

In [ ]:
vocabulary, inverse_vocabulary = let

    # initialize -
    vocabulary = Dict{String, Int64}();
    inverse_vocabulary = Dict{Int64, String}();
    index = 1;

    # control tokens -
    control_tokens = ["<bos>", "<eos>", "<pad>", "<unk>"];

    # tmp variables -
    words = Set{String}();
    for sentence in sentences
        tmp = split(lowercase(sentence));
        push!(words, tmp...);
    end
    words_array = collect(words) |> sort;

    # append the control tokens to the words array
    words_array = vcat(words_array, control_tokens);
    for word in words_array
        vocabulary[word] = index;
        inverse_vocabulary[index] = word;
        index += 1;
    end

    # return the vocabulary and inverse vocabulary
    (vocabulary, inverse_vocabulary)
end

___

## Task 1: Build Weighted Co-occurrence Matrix
GloVe operates on a global word-word co-occurrence matrix rather than individual (context, target) training pairs. We construct this matrix by sliding a window over each sentence and accumulating distance-weighted counts.

> __Distance-weighted co-occurrence__
>
> For each center word at position $t$ and a context word at position $t + d$ (where $1 \leq |d| \leq m$ for window size $m$), we add $1/|d|$ to the co-occurrence count $X_{ij}$. Closer words receive higher weight because they are more likely to be syntactically or semantically related. This produces a symmetric matrix $\mathbf{X}\in\mathbb{R}^{N_{\mathcal{V}}\times N_{\mathcal{V}}}$ where each entry reflects the total weighted co-occurrence between two words across the corpus.

We store the result in the `X::Array{Float64, 2}` variable by calling the `build_weighted_cooccurrence_matrix` function.

In [ ]:
X = build_weighted_cooccurrence_matrix(sentences, vocabulary, window_size = 2);
println("Co-occurrence matrix size: $(size(X)), non-zero entries: $(count(X .> 0))")

Let's inspect a few entries to verify the co-occurrence counts.

In [ ]:
let
    check_pairs = [("machine", "learning"), ("data", "science"), ("fun", "great"), ("machine", "?")];
    for (w1, w2) in check_pairs
        i = vocabulary[w1];
        j = vocabulary[w2];
        println("X[$(w1), $(w2)] = $(round(X[i, j], digits=4))");
    end
end

___

## Task 2: Train GloVe Embeddings
With the co-occurrence matrix in hand, we train GloVe embeddings by minimizing a weighted least-squares objective over all non-zero entries.

> __GloVe objective__
>
> For each word pair $(i,j)$ where $X_{ij} > 0$, the GloVe loss is: $\mathcal{J} = \sum_{i,j} f(X_{ij})\left(\mathbf{w}_i^\top \mathbf{w}_{\text{ctx},j} + b_i + b_{\text{ctx},j} - \log X_{ij}\right)^2$ where $f(X_{ij}) = \min\left(1, (X_{ij}/x_{\max})^\alpha\right)$ is a weighting function that prevents very frequent co-occurrences from dominating the loss. The parameters $\mathbf{w}_i, \mathbf{w}_{\text{ctx},j}\in\mathbb{R}^d$ are word and context embedding vectors, and $b_i, b_{\text{ctx},j}\in\mathbb{R}$ are scalar bias terms.

We call `train_glove` and store the results in five variables: `W_word::Matrix{Float64}` (word embeddings of shape $d \times N_{\mathcal{V}}$), `W_ctx::Matrix{Float64}` (context embeddings), `b_word::Vector{Float64}` (word biases), `b_ctx::Vector{Float64}` (context biases), and `loss_history::Vector{Float64}` (per-epoch average loss).

In [ ]:
W_word, W_ctx, b_word, b_ctx, loss_history = train_glove(X, length(vocabulary), d = 5, η = 0.05, num_epochs = 500);
println("Training complete. Final average loss: $(round(loss_history[end], digits=4))")

Let's plot the training loss to verify that the model converges.

In [ ]:
let
    plot(loss_history, xlabel="Epoch", ylabel="Average Loss", label="GloVe Training Loss", lw=2)
end

___

## Task 3: Compare Embeddings
The GloVe authors recommend combining the word and context vectors as the final embedding, since both capture useful information. We compute cosine similarities for the same word pairs used in L9a and L9c so the student can visually compare GloVe to CBOW and Skip-Gram.

> __Combining word and context vectors__
>
> Because GloVe treats word and context vectors symmetrically, the final embedding for word $i$ is $\mathbf{e}_i = \mathbf{w}_i + \mathbf{w}_{\text{ctx},i}$. This combination leverages information from both roles (center and context) and typically produces better embeddings than using either vector alone. We measure similarity using cosine similarity: $\cos(\mathbf{e}_i, \mathbf{e}_j) = \mathbf{e}_i \cdot \mathbf{e}_j / (\|\mathbf{e}_i\|_2 \|\mathbf{e}_j\|_2)$.

We store the combined embeddings in the `embeddings::Matrix{Float64}` variable of shape $d \times N_{\mathcal{V}}$.

In [ ]:
embeddings = W_word + W_ctx;
println("Combined embedding matrix size: $(size(embeddings))")

Now let's compute cosine similarities for selected word pairs and display the results in a table.

In [ ]:
let
    function cosine_sim(w1::String, w2::String, E::Matrix{Float64}, vocab::Dict{String, Int64})
        v1 = E[:, vocab[w1]];
        v2 = E[:, vocab[w2]];
        return dot(v1, v2) / (norm(v1) * norm(v2));
    end

    word_pairs = [
        ("machine", "learning"),
        ("data", "science"),
        ("fun", "great"),
        ("love", "enjoy"),
        ("machine", "?"),
    ];

    # build the results table -
    pair_labels = ["$(w1) / $(w2)" for (w1, w2) in word_pairs];
    similarities = [round(cosine_sim(w1, w2, embeddings, vocabulary), digits=4) for (w1, w2) in word_pairs];

    df = DataFrame("Word Pair" => pair_labels, "GloVe Cosine Similarity" => similarities);
    pretty_table(df, header=["Word Pair", "GloVe Cosine Similarity"])
end

___

## Summary
In this example, we trained GloVe embeddings from scratch on a toy corpus by constructing a weighted co-occurrence matrix and minimizing a weighted least-squares objective.

> __Key Takeaways__:
> 
> * __Distance-weighted co-occurrence captures global statistics__: The co-occurrence matrix accumulates $1/d$ weights for each word pair within a sliding window across the entire corpus. This global counting approach contrasts with prediction-based methods like CBOW and Skip-Gram, which process local context windows one at a time.
> * __Weighted least-squares training avoids softmax__: GloVe minimizes a regression-style loss over non-zero co-occurrence entries with a capping function $f(X_{ij})$ that limits the influence of very frequent pairs. This avoids the expensive softmax normalization required by CBOW and Skip-Gram.
> * __Combined word and context vectors form the final embedding__: Because GloVe treats center and context roles symmetrically, adding the two learned vectors for each word produces a single embedding. The student can compare the cosine similarities from this table to the CBOW and Skip-Gram results in L9a and L9c.

These results show how a count-based factorization method produces word embeddings that can be directly compared to those from prediction-based models.
___